# Language-model evaluation (perplexity, BLEU, ROUGE, METEOR) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Evaluate generated text by likelihood, overlap, and alignment-aware similarity
The cells compute cross-entropy/perplexity, BLEU precision, ROUGE recall, a METEOR-style F score, and show metric disagreement.

In [ ]:
probs=np.array([0.5,0.25,0.25]); ce=-np.mean(np.log(probs)); ppl=math.exp(ce)
plt.figure(figsize=(5,3)); plt.bar(['cross-entropy','perplexity'],[ce,ppl]); plt.title('Perplexity exponentiates average NLL')
assert abs(ppl-3.174802103936399)<1e-9

In [ ]:
cand='the cat sat'.split(); ref='the cat slept'.split(); prec=sum(w in ref for w in cand)/len(cand)
plt.figure(figsize=(4,3)); plt.bar(['BLEU-1 precision'],[prec]); plt.ylim(0,1); plt.title('BLEU rewards candidate precision')
assert abs(prec-2/3)<1e-9

In [ ]:
cand='the cat sat'.split(); ref='the small cat sat'.split(); recall=sum(w in cand for w in ref)/len(ref)
plt.figure(figsize=(4,3)); plt.bar(['ROUGE-1 recall'],[recall]); plt.ylim(0,1); plt.title('ROUGE rewards covering reference words')
assert abs(recall-0.75)<1e-9

In [ ]:
P=2/3; R=2/4; meteor=10*P*R/(R+9*P)
plt.figure(figsize=(4,3)); plt.bar(['P','R','METEOR'],[P,R,meteor]); plt.ylim(0,1); plt.title('METEOR weights recall heavily')
assert abs(meteor-0.5128205128205128)<1e-9

In [ ]:
models=['fluent short','faithful long']; bleu=[0.8,0.55]; rouge=[0.4,0.85]
plt.figure(figsize=(5,3)); x=np.arange(2); plt.bar(x-.15,bleu,.3,label='BLEU'); plt.bar(x+.15,rouge,.3,label='ROUGE'); plt.xticks(x,models); plt.legend(); plt.title('Metrics can disagree')
assert np.argmax(bleu)!=np.argmax(rouge)